# Encoding the Molecular Hamiltonian 

This notebook shows how to use a Fermion-Qubit encoding to encode a second quantised fermionic hamiltonian.

For the time being, the only fermionic hamiltonian we can encode is the molecular hamiltonian which describes chemistry. (If you are reading this and want us to support something else - just ask!).

## Simple Useage

Let's first get the coefficients for our second quantised hamiltonian.

For the time being we can use randomly generated ones.

In [28]:
import numpy as np
constant_energy = 0
one_e_coeffs = np.random.random((3,3))
two_e_coeffs = np.random.random((3,3,3,3))


Rather than figure out how many modes we need for an encoding, we can create one of the right size by passing in our coefficients.

In [29]:
from ferrmion import TernaryTree
tt = TernaryTree.from_hamiltonian_coefficients((one_e_coeffs, two_e_coeffs))

To encode a hamiltonian, we need both a mapping from fermonic operators to qubits, and an enumeration scheme which accounts for the degrees of freedom when labeling modes and qubits.

For the mapping, we'll use JordanWigner and for the enumeration scheme, we can use the default one.

Note we can [optimise our encoded hamiltonian](https://ferrmion.readthedocs.io/en/latest/notebooks/pauli_weight.html) by getting more clever with these.

In [30]:
jw = tt.JW()
jw.enumeration_scheme = jw.default_enumeration_scheme()
jw.enumeration_scheme

{'': (0, 0), 'z': (1, 1), 'zz': (2, 2)}

In [31]:
from ferrmion.hamiltonians import molecular_hamiltonian
hamiltonian = molecular_hamiltonian(encoding=jw, one_e_coeffs=one_e_coeffs, two_e_coeffs=two_e_coeffs, constant_energy=constant_energy)
hamiltonian

{'ZZI': (-0.20815406753720697+0j),
 'YXI': 0.11449347538585583j,
 'IXX': (0.28883423017036514+0j),
 'YYI': (0.23636677189475025+0j),
 'ZYY': (0.07887582447841886+0j),
 'ZYX': -0.12292952463720253j,
 'XZY': 0.47142672294871585j,
 'XIX': (-0.04526557168697053+0j),
 'XYI': -0.11449347538585584j,
 'XXZ': (0.06991599669859612+0j),
 'XYZ': 0.061600030143413226j,
 'XIY': -0.2757679105407842j,
 'IYY': (0.2888342301703652+0j),
 'YIY': (-0.04526557168697054+0j),
 'IZZ': (-0.10714931334118502+0j),
 'IYX': 0.16284676452297125j,
 'YYZ': (0.06991599669859609+0j),
 'ZXY': 0.12292952463720254j,
 'XZX': (0.24179882465683838+0j),
 'ZII': (-0.06454442223226381+0j),
 'YZX': -0.4714267229487159j,
 'YXZ': -0.06160003014341321j,
 'XXI': (0.2363667718947504+0j),
 'IZI': (0.29903382328328165+0j),
 'YZY': (0.2417988246568384+0j),
 'IIZ': (-0.328503320484393+0j),
 'IXY': -0.16284676452297125j,
 'III': (0.3222673871786505+0j),
 'YIX': 0.2757679105407842j,
 'ZXX': (0.07887582447841884+0j),
 'ZIZ': (0.0870499131331

## Hamiltonian Templates

Sometimes it can be useful to find out how an encoding behaves without providing coefficients. This lets us see which terms of the second quantised hamiltonian contribute to which pauli terms of the qubit Hamiltonian.

### Creating a Template

In [32]:
from ferrmion.hamiltonians import molecular_hamiltonian_template

The only information we need to provide is a set of XZ-encoded vectors (this is how ferrmion manipulates encodings internally) and imaginary factors for these vectors. 

In [33]:
ipowers, symplectics = jw._build_symplectic_matrix()

In [34]:
ipowers

array([0, 1, 0, 1, 0, 1], dtype=uint8)

In [35]:
np.array(symplectics, dtype=int)

array([[1, 0, 0, 0, 0, 0],
       [1, 0, 0, 1, 0, 0],
       [0, 1, 0, 1, 0, 0],
       [0, 1, 0, 1, 1, 0],
       [0, 0, 1, 1, 1, 0],
       [0, 0, 1, 1, 1, 1]])

In [36]:
template = molecular_hamiltonian_template(ipowers, symplectics)
template

{'XYZ': {(1, 2, 0, 2): -0.125j,
  (1, 2, 2, 0): 0.125j,
  (2, 1, 2, 0): -0.125j,
  (0, 2, 1, 2): 0.125j,
  (2, 0, 2, 1): 0.125j,
  (2, 1, 0, 2): 0.125j,
  (0, 2, 2, 1): -0.125j,
  (2, 0, 1, 2): -0.125j},
 'XIX': {(0, 1, 1, 2): (-0.125+0j),
  (1, 0, 2, 1): (-0.125+0j),
  (2, 1, 0, 1): (0.125+0j),
  (1, 2, 1, 0): (0.125+0j),
  (1, 0, 1, 2): (0.125+0j),
  (0, 1, 2, 1): (0.125+0j),
  (2, 1, 1, 0): (-0.125+0j),
  (1, 2, 0, 1): (-0.125+0j)},
 'IYX': {(0, 2, 1, 0): 0.125j,
  (0, 1, 0, 2): 0.125j,
  (1, 0, 2, 0): 0.125j,
  (1, 2): -0.25j,
  (1, 0, 0, 2): -0.125j,
  (2, 0, 0, 1): 0.125j,
  (2, 0, 1, 0): -0.125j,
  (0, 1, 2, 0): -0.125j,
  (0, 2, 0, 1): -0.125j,
  (2, 1): 0.25j},
 'XXZ': {(2, 1, 2, 0): (0.125+0j),
  (2, 0, 1, 2): (-0.125+0j),
  (1, 2, 0, 2): (0.125+0j),
  (0, 2, 2, 1): (-0.125+0j),
  (2, 1, 0, 2): (-0.125+0j),
  (1, 2, 2, 0): (-0.125+0j),
  (0, 2, 1, 2): (0.125+0j),
  (2, 0, 2, 1): (0.125+0j)},
 'YZX': {(0, 2): -0.25j,
  (2, 0): 0.25j,
  (2, 1, 1, 0): 0.125j,
  (0, 1, 1, 2): -0.

### Filling a Template

So we can now see which modes contribute to each Pauli operator.

We can now fill out our template with the coefficients we used earlier.

This time we only need to provide a mapping from fermionic modes to majorana operators. Again we can use the default.

In [37]:
jw.default_mode_op_map

{0: 0, 1: 1, 2: 2}

In [38]:
from ferrmion.hamiltonians import fill_template
filled_template = fill_template(template,constant_energy=constant_energy, one_e_terms=one_e_coeffs, two_e_terms=two_e_coeffs, mode_op_map=jw.default_mode_op_map)
filled_template

{'XZY': 0.47142672294871585j,
 'IZI': (0.29903382328328165+0j),
 'ZII': (-0.06454442223226381+0j),
 'IIZ': (-0.328503320484393+0j),
 'III': (0.3222673871786504+0j),
 'IYX': 0.1628467645229712j,
 'XIY': -0.2757679105407842j,
 'IXY': -0.16284676452297123j,
 'YYI': (0.23636677189475033+0j),
 'XXI': (0.23636677189475033+0j),
 'YZY': (0.24179882465683838+0j),
 'YZX': -0.4714267229487159j,
 'IZZ': (-0.10714931334118502+0j),
 'YIY': (-0.04526557168697053+0j),
 'XIX': (-0.045265571686970554+0j),
 'XYI': -0.11449347538585586j,
 'XZX': (0.24179882465683833+0j),
 'YXZ': -0.06160003014341327j,
 'IYY': (0.28883423017036525+0j),
 'ZXX': (0.07887582447841886+0j),
 'YXI': 0.11449347538585582j,
 'ZZI': (-0.20815406753720697+0j),
 'YIX': 0.2757679105407842j,
 'YYZ': (0.06991599669859608+0j),
 'IXX': (0.28883423017036514+0j),
 'ZIZ': (0.08704991313311675+0j),
 'XXZ': (0.06991599669859608+0j),
 'ZXY': 0.12292952463720253j,
 'XYZ': 0.061600030143413254j,
 'ZYX': -0.12292952463720253j,
 'ZYY': (0.0788758244

Let' check to see that we have the same hamiltonian in each.

In [39]:
assert hamiltonian.keys() == filled_template.keys()
for k in hamiltonian.keys():
    # There is a little numerical instability so 
    # some values are different by 1e-18 or so!
    assert np.isclose(hamiltonian[k], filled_template[k])